# Online Pipeline: Inference & Submission (BM25 + FAISS Hybrid)
Notebook này thực hiện bước Inference để tạo file `submission.csv`.
Nó sẽ sử dụng các index đã được build từ bước Offline (`03_build_index.ipynb`) để chạy cực kỳ nhanh chóng.


In [ ]:
!pip install -q pandas pyarrow rank_bm25 sentence-transformers faiss-cpu faiss-gpu bitsandbytes accelerate


In [ ]:
import pandas as pd
import pickle
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
import torch
import warnings
warnings.filterwarnings('ignore')

print("Đang tải các file Index (BM25, FAISS, Metadata)...")
# Tải Metadata
df_chunks = pd.read_parquet('chunk_metadata.parquet')
print(f"Đã tải {len(df_chunks)} chunks metadata.")

# Tải BM25 Index
with open('bm25_index.pkl', 'rb') as f:
    bm25 = pickle.load(f)
print("Đã tải BM25 Index.")

# Tải FAISS Index
faiss_index = faiss.read_index('faiss_index.bin')
print(f"Đã tải FAISS Index chứa {faiss_index.ntotal} vectors.")

# Load lại mô hình Qwen
print("Khởi tạo Qwen Embedding Model (4-bit)...")
from transformers import BitsAndBytesConfig
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)
eval_model = SentenceTransformer(
    "Qwen/Qwen3-Embedding-8B", 
    model_kwargs={'device': 'cuda', 'quantization_config': quantization_config},
    trust_remote_code=True
)
eval_model.max_seq_length = 512


In [ ]:
# Thuật toán Reciprocal Rank Fusion (RRF)
def rrf(rank_list_1, rank_list_2, k=60):
    rrf_scores = {}
    
    for rank, doc_idx in enumerate(rank_list_1):
        rrf_scores[doc_idx] = rrf_scores.get(doc_idx, 0.0) + 1.0 / (k + rank + 1)
        
    for rank, doc_idx in enumerate(rank_list_2):
        rrf_scores[doc_idx] = rrf_scores.get(doc_idx, 0.0) + 1.0 / (k + rank + 1)
        
    # Sort theo RRF score giảm dần
    sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return [doc_idx for doc_idx, score in sorted_docs]

# Hàm tìm kiếm cho 1 câu query
def hybrid_search(query, top_k=10, faiss_k=50, bm25_k=50):
    # 1. FAISS Search
    query_emb = eval_model.encode([query], normalize_embeddings=True)
    distances, faiss_indices = faiss_index.search(query_emb, faiss_k)
    faiss_top_indices = faiss_indices[0].tolist()
    
    # 2. BM25 Search
    tokenized_query = query.lower().split(" ")
    bm25_scores = bm25.get_scores(tokenized_query)
    bm25_top_indices = np.argsort(bm25_scores)[::-1][:bm25_k].tolist()
    
    # 3. Kết hợp bằng RRF
    fused_indices = rrf(faiss_top_indices, bm25_top_indices)
    
    # Lấy top_k doc_id KHÔNG TRÙNG LẶP từ fused_indices
    result_doc_ids = []
    seen_doc_ids = set()
    
    for idx in fused_indices:
        doc_id = df_chunks.iloc[idx]['doc_id']
        if doc_id not in seen_doc_ids:
            seen_doc_ids.add(doc_id)
            result_doc_ids.append(doc_id)
            if len(result_doc_ids) == top_k:
                break
                
    return result_doc_ids


In [ ]:
# Đọc file Query
df_queries = pd.read_parquet('/kaggle/input/ten-dataset-cua-ban/query.parquet') # Đổi lại đường dẫn phù hợp
print(f"Tổng số truy vấn: {len(df_queries)}")

# Chạy Inference
results = []
for idx, row in df_queries.iterrows():
    q_id = row['query_id']
    q_text = row['query']
    
    # Lấy 10 doc_id tốt nhất
    top_10_docs = hybrid_search(q_text, top_k=10)
    
    # Format lại thành string cách nhau bởi dấu phẩy
    relevant_docs_str = ",".join(top_10_docs)
    
    results.append({
        'query_id': q_id,
        'relevant_docs': relevant_docs_str
    })
    
    if (idx + 1) % 100 == 0:
        print(f"Đã xử lý {idx + 1}/{len(df_queries)} truy vấn...")

# Lưu file Submission
df_sub = pd.DataFrame(results)
df_sub.to_csv('submission.csv', index=False)
print("Đã tạo file submission.csv thành công!")

# Nén thành file ZIP (để nộp Kaggle)
import zipfile
with zipfile.ZipFile('submission.zip', 'w') as zf:
    zf.write('submission.csv')
print("Đã nén thành submission.zip. BẠN CÓ THỂ NỘP FILE NÀY LÊN HỆ THỐNG!")
